In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning) # 抑制分箱时的警告

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# 设置绘图风格
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid", font='SimHei')

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """从 DWD 表获取有数据的非空城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_price_data(city_name):
    """根据所选城市获取该城市全量总价数据"""
    conn = get_db_connection()
    # 使用参数化查询防止注入，同时取全部分区数据
    query = """
    SELECT total_price 
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city = %s AND total_price IS NOT NULL
    """
    try:
        df = pd.read_sql(query, conn, params=(city_name,))
        return df['total_price'].values
    finally:
        conn.close()

In [2]:
def plot_price_stairs(city):
    """绘制特定城市的阶梯填充图"""
    prices = fetch_price_data(city)
    
    if len(prices) == 0:
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, f"{city} 暂无房价数据", ha='center', va='center', fontsize=15)
        plt.axis('off')
        plt.show()
        return

    # --- 自适应区间逻辑 ---
    # 截断 1% 和 99% 的极端值，让图表聚焦于主流区间，避免被超高价豪宅拉伸
    p1 = np.percentile(prices, 1)
    p99 = np.percentile(prices, 99)
    filtered_prices = prices[(prices >= p1) & (prices <= p99)]
    
    # 动态计算分箱数 (Freedman-Diaconis rule 或 固定数量)
    # 这里我们固定划分为 40 个阶梯，以保证设计感的连贯性
    counts, bins = np.histogram(filtered_prices, bins=40)

    # --- 开始绘图 ---
    fig, ax = plt.subplots(figsize=(14, 7))

    # 1. 绘制阶梯图的“填充”区域 (底层)
    # 使用浅蓝色带透明度，营造呼吸感
    ax.stairs(counts, bins, fill=True, alpha=0.35, color='#4C72B0')
    
    # 2. 绘制阶梯图的“边缘线” (顶层)
    # 使用加深的同系色，勾勒出明确的阶梯感
    ax.stairs(counts, bins, fill=False, linewidth=2.5, color='#2C4B7E')

    # --- 辅助线与高亮 ---
    # 计算并标出中位数
    median_price = np.median(filtered_prices)
    ax.axvline(median_price, color='#D55E00', linestyle='--', linewidth=2, alpha=0.8)
    ax.text(median_price + (p99-p1)*0.02, max(counts)*0.9, 
            f'中位数: {median_price:.1f}万', color='#D55E00', fontweight='bold', fontsize=12)

    # 图表修饰
    plt.title(f'[{city}] 二手房总价区间分布特征 (阶梯填充图)', fontsize=16, pad=20)
    plt.xlabel('房屋总价 (万元)', fontsize=12)
    plt.ylabel('房源数量 (套)', fontsize=12)
    
    # 美化坐标轴
    sns.despine(left=True)
    ax.grid(axis='y', linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

In [3]:
def render_interactive_dashboard():
    """构建交互式组件并渲染"""
    cities = fetch_cities()
    if not cities:
        print("未获取到城市列表，请检查数据表。")
        return
    
    # 将北京或列表第一个城市设为默认值
    default_city = '北京' if '北京' in cities else cities[0]
    
    # 1. 创建下拉组件
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 切换城市:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px', margin='0 0 20px 0')
    )
    
    # 2. 创建输出区域组件
    plot_output = widgets.Output()

    # 3. 定义回调函数
    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            # clear_output 配合 wait=True 可以实现无缝刷新，避免白屏闪烁
            with plot_output:
                clear_output(wait=True)
                plot_price_stairs(change['new'])

    # 4. 绑定监听器
    city_dropdown.observe(on_city_change)

    # 5. 首次展示组件和默认图表
    display(city_dropdown)
    display(plot_output)
    
    # 触发默认城市的绘图
    with plot_output:
        plot_price_stairs(default_city)

In [4]:
if __name__ == "__main__":
    # 执行渲染
    render_interactive_dashboard()

Dropdown(description='📍 切换城市:', index=4, layout=Layout(margin='0 0 20px 0', width='300px'), options=('上海', '东莞…

Output()